# Chapter 8: Data Governance in Practice
## CompTIA SecAI+ Study Lab

**Objectives:**
- Detect and mask Personally Identifiable Information (PII)
- Implement Role-Based Access Control (RBAC) simulation
- Apply data retention policies programmatically
- Perform a data quality audit across multiple dimensions
- Build a governance audit report

> **Exam Tip:** CompTIA SecAI+ defines **data governance** as the set of policies, standards, and processes that ensure data is managed as a valuable asset. Key governance concepts: **data stewardship**, **data catalog**, **data lineage**, **master data management (MDM)**, **PII/PHI**, and **RBAC**. These are frequently tested.


---
## Exercise 1: PII Detection

**Personally Identifiable Information (PII)** is any data that can identify an individual. Common PII fields:
- Name, email, phone number
- Social Security Number (SSN), passport number
- Date of birth, home address, IP address

> **Exam Tip:** Know the difference between **PII** (general: name, address, email) and **PHI** (Protected Health Information under HIPAA: medical records, diagnoses). Also know **PCI DSS** (payment card data). The exam tests which regulatory framework applies to which data type.


In [ ]:
import pandas as pd
import re

# Sample dataset with various column types
employees = pd.DataFrame({
    'employee_id':  [1001, 1002, 1003, 1004, 1005],
    'full_name':    ['Alice Johnson', 'Bob Smith', 'Carol White', 'Dave Brown', 'Eve Davis'],
    'email':        ['alice@company.com', 'bob@company.com', 'carol@co.org', 'dave@company.com', 'eve@co.net'],
    'ssn':          ['123-45-6789', '987-65-4321', '456-78-9012', '321-54-9876', '654-32-1098'],
    'phone':        ['(512) 555-0101', '(303) 555-0202', '(415) 555-0303', '(212) 555-0404', '(713) 555-0505'],
    'department':   ['Engineering', 'HR', 'Marketing', 'Finance', 'Engineering'],
    'salary':       [95000, 72000, 78000, 88000, 91000],
    'hire_date':    ['2020-03-15', '2019-07-01', '2021-11-20', '2018-05-10', '2022-01-30'],
    'ip_address':   ['192.168.1.10', '10.0.0.25', '172.16.0.5', '192.168.2.30', '10.1.1.100']
})

# TODO: Write a function detect_pii_columns(df) that:
#   1. Checks column NAMES against a list of known PII keywords
#      (name, email, ssn, phone, address, dob, ip, passport, birth)
#   2. Also uses regex patterns to detect PII-like VALUES in string columns:
#      - Email pattern: contains '@'
#      - SSN pattern: XXX-XX-XXXX
#      - Phone pattern: digits, dashes, parentheses, spaces
#      - IP pattern: X.X.X.X
#   3. Returns a dict: {column_name: [reasons]}

def detect_pii_columns(df):
    pii_keywords = ['name', 'email', 'ssn', 'phone', 'address', 'dob', 'ip', 'passport', 'birth', 'social']
    pii_patterns = {
        'email':   r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}',
        'ssn':     r'\d{3}-\d{2}-\d{4}',
        'phone':   r'\(?\d{3}\)?[\s\-\.]\d{3}[\s\-\.]\d{4}',
        'ip_addr': r'\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}'
    }

    results = {}

    for col in df.columns:
        reasons = []
        # TODO: Check if column name contains a PII keyword

        # TODO: For string/object columns, sample up to 10 values
        #       and test each PII regex pattern against them

        if reasons:
            results[col] = reasons

    return results

pii_found = detect_pii_columns(employees)
print("PII Column Detection Results:")
for col, reasons in (pii_found or {}).items():
    print(f"  [{col}]: {', '.join(reasons)}")


In [ ]:
# SOLUTION
# import re
#
# def detect_pii_columns(df):
#     pii_keywords = ['name', 'email', 'ssn', 'phone', 'address', 'dob', 'ip', 'passport', 'birth', 'social']
#     pii_patterns = {
#         'email':   r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}',
#         'ssn':     r'\d{3}-\d{2}-\d{4}',
#         'phone':   r'\(?\d{3}\)?[\s\-\.]\d{3}[\s\-\.]\d{4}',
#         'ip_addr': r'\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}'
#     }
#     results = {}
#     for col in df.columns:
#         reasons = []
#         col_lower = col.lower()
#         for keyword in pii_keywords:
#             if keyword in col_lower:
#                 reasons.append(f'name matches keyword "{keyword}"')
#                 break
#         if df[col].dtype == object:
#             sample = df[col].dropna().astype(str).head(10)
#             for pattern_name, pattern in pii_patterns.items():
#                 if sample.apply(lambda x: bool(re.search(pattern, x))).any():
#                     if pattern_name not in str(reasons):
#                         reasons.append(f'value matches {pattern_name} pattern')
#         if reasons:
#             results[col] = list(set(reasons))
#     return results
#
# pii_found = detect_pii_columns(employees)
# print("PII Column Detection Results:")
# for col, reasons in pii_found.items():
#     print(f"  [{col}]: {', '.join(reasons)}")


---
## Exercise 2: Data Masking

**Data masking** replaces real sensitive values with realistic but fictional ones, allowing non-production use (testing, development, analytics) without exposing actual PII.

Masking techniques:
- **Substitution** — Replace with a random value of the same type (Faker library)
- **Redaction** — Replace with a placeholder (e.g., `****`)
- **Tokenization** — Replace with a token that maps back to the real value (stored separately)
- **Encryption** — Mathematically transform the value

> **Exam Tip:** Know the difference between masking (one-way, cannot recover original) and tokenization (two-way, can recover with token map). Encryption is also reversible with the key. The exam may ask which method is appropriate for different regulatory scenarios.


In [ ]:
import pandas as pd
import re
import random
import string

# Note: If 'faker' is installed, use it. Otherwise we'll simulate basic masking.
try:
    from faker import Faker
    fake = Faker()
    FAKER_AVAILABLE = True
except ImportError:
    FAKER_AVAILABLE = False
    print("Faker not installed. Install with: pip install faker")
    print("Using basic substitution masking instead.")

df_original = employees.copy()

# TODO: Write a function mask_dataframe(df, columns_to_mask) that:
#   - Creates a masked copy of the dataframe
#   - For 'full_name': replace with fake name (Faker) or 'Person_XXX'
#   - For 'email': replace with fake email or 'masked_XXX@example.com'
#   - For 'ssn': replace with 'XXX-XX-XXXX'
#   - For 'phone': replace with '(000) 000-0000'
#   - For 'ip_address': replace with '0.0.0.0'
#   Returns the masked DataFrame

def mask_dataframe(df, columns_to_mask):
    masked = df.copy()
    for col in columns_to_mask:
        if col not in masked.columns:
            continue
        # TODO: apply masking per column type
        pass
    return masked

pii_columns = ['full_name', 'email', 'ssn', 'phone', 'ip_address']
df_masked = mask_dataframe(df_original, pii_columns)

print("Original:")
print(df_original[pii_columns].to_string())
print("\nMasked:")
print(df_masked[pii_columns].to_string())


In [ ]:
# SOLUTION
# import pandas as pd
# import re, random, string
#
# try:
#     from faker import Faker
#     fake = Faker()
#     FAKER_AVAILABLE = True
# except ImportError:
#     FAKER_AVAILABLE = False
#
# def mask_dataframe(df, columns_to_mask):
#     masked = df.copy()
#     for i, col in enumerate(columns_to_mask):
#         if col not in masked.columns:
#             continue
#         n = len(masked)
#         if col == 'full_name':
#             masked[col] = [fake.name() if FAKER_AVAILABLE else f'Person_{j+1:03d}' for j in range(n)]
#         elif col == 'email':
#             masked[col] = [fake.email() if FAKER_AVAILABLE else f'masked_{j+1:03d}@example.com' for j in range(n)]
#         elif col == 'ssn':
#             masked[col] = 'XXX-XX-XXXX'
#         elif col == 'phone':
#             masked[col] = '(000) 000-0000'
#         elif col == 'ip_address':
#             masked[col] = '0.0.0.0'
#         else:
#             masked[col] = '****'
#     return masked
#
# pii_columns = ['full_name', 'email', 'ssn', 'phone', 'ip_address']
# df_masked = mask_dataframe(employees.copy(), pii_columns)
# print("Original:")
# print(employees[pii_columns].to_string())
# print("\nMasked:")
# print(df_masked[pii_columns].to_string())


---
## Exercise 3: Simulate Role-Based Access Control (RBAC)

**RBAC** restricts data access based on the user's role. Each role has a set of permitted actions on specific resources.

Example roles:
- **Admin** — full access to all data
- **Analyst** — read access to non-PII data
- **HR_Manager** — read/write access to HR data including salaries
- **Auditor** — read-only access to audit logs

> **Exam Tip:** Know **RBAC** (roles), **ABAC** (attributes, more fine-grained), and **DAC** (discretionary — owner controls access). RBAC is the most common model tested. Also know the **Principle of Least Privilege**: grant only the minimum access needed.


In [ ]:
# RBAC Permission System Simulation

# Define permission matrix
PERMISSIONS = {
    'admin': {
        'employee_data':    ['read', 'write', 'delete'],
        'financial_data':   ['read', 'write', 'delete'],
        'pii_data':         ['read', 'write', 'delete'],
        'audit_logs':       ['read', 'write']
    },
    'analyst': {
        'employee_data':    ['read'],
        'financial_data':   ['read'],
        'pii_data':         [],
        'audit_logs':       []
    },
    'hr_manager': {
        'employee_data':    ['read', 'write'],
        'financial_data':   [],
        'pii_data':         ['read'],
        'audit_logs':       ['read']
    },
    'auditor': {
        'employee_data':    ['read'],
        'financial_data':   ['read'],
        'pii_data':         [],
        'audit_logs':       ['read']
    }
}

# TODO: Write a function check_permission(user_role, resource, action) that:
#   - Returns True if the role has the permission
#   - Returns False if not
#   - Logs the access attempt (print a formatted access log line)
#   Format: [ALLOW/DENY] role=X | resource=Y | action=Z

def check_permission(user_role, resource, action):
    pass  # TODO

# TODO: Write a function get_allowed_columns(user_role, df) that:
#   - Returns a filtered DataFrame with only columns the role can see
#   - Analysts cannot see PII columns (ssn, phone, ip_address, full_name)
#   - HR managers can see all but salary
#   - Admins see everything

def get_allowed_columns(user_role, df):
    pii_cols = ['full_name', 'email', 'ssn', 'phone', 'ip_address']
    restricted_financial = ['salary']

    if user_role == 'admin':
        return df
    elif user_role == 'analyst':
        # TODO: return df without pii_cols
        pass
    elif user_role == 'hr_manager':
        # TODO: return df without restricted_financial
        pass
    elif user_role == 'auditor':
        # TODO: return df with only employee_id, department, hire_date
        pass
    else:
        return pd.DataFrame()  # No access

# Test
print("=== Access Control Tests ===")
check_permission('analyst', 'pii_data', 'read')
check_permission('analyst', 'pii_data', 'write')
check_permission('hr_manager', 'employee_data', 'write')
check_permission('auditor', 'financial_data', 'delete')

print("\n=== Column Access by Role ===")
for role in ['admin', 'analyst', 'hr_manager', 'auditor']:
    result = get_allowed_columns(role, employees)
    if result is not None:
        cols = list(result.columns) if hasattr(result, 'columns') else []
        print(f"  {role:12s}: {cols}")


In [ ]:
# SOLUTION
# from datetime import datetime
#
# def check_permission(user_role, resource, action):
#     role_perms = PERMISSIONS.get(user_role, {})
#     resource_perms = role_perms.get(resource, [])
#     allowed = action in resource_perms
#     status = 'ALLOW' if allowed else 'DENY '
#     ts = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
#     print(f"[{ts}] [{status}] role={user_role:12s} | resource={resource:20s} | action={action}")
#     return allowed
#
# def get_allowed_columns(user_role, df):
#     pii_cols = ['full_name', 'email', 'ssn', 'phone', 'ip_address']
#     restricted_financial = ['salary']
#     if user_role == 'admin':
#         return df
#     elif user_role == 'analyst':
#         return df.drop(columns=[c for c in pii_cols if c in df.columns])
#     elif user_role == 'hr_manager':
#         return df.drop(columns=[c for c in restricted_financial if c in df.columns])
#     elif user_role == 'auditor':
#         allowed = [c for c in ['employee_id', 'department', 'hire_date'] if c in df.columns]
#         return df[allowed]
#     return pd.DataFrame()
#
# print("=== Access Control Tests ===")
# check_permission('analyst', 'pii_data', 'read')
# check_permission('analyst', 'pii_data', 'write')
# check_permission('hr_manager', 'employee_data', 'write')
# check_permission('auditor', 'financial_data', 'delete')
#
# print("\n=== Column Access by Role ===")
# for role in ['admin', 'analyst', 'hr_manager', 'auditor']:
#     result = get_allowed_columns(role, employees)
#     print(f"  {role:12s}: {list(result.columns)}")


---
## Exercise 4: Data Retention Policy

**Data retention** defines how long data should be kept before it must be deleted or archived. Regulations like GDPR require data to not be kept longer than necessary.

Common retention rules:
- Financial records: 7 years
- Employee records: 7 years after termination
- Customer PII: delete within 30 days of opt-out request

> **Exam Tip:** Know **GDPR** (European, includes right to erasure / right to be forgotten), **CCPA** (California), **HIPAA** (US healthcare), **SOX** (US financial records). Each has specific retention requirements. The exam may present a scenario and ask which regulation applies.


In [ ]:
import pandas as pd
from datetime import datetime, timedelta

# Customer records with creation dates
today = datetime(2024, 12, 1)
customer_records = pd.DataFrame({
    'customer_id': range(1001, 1016),
    'name':        [f'Customer_{i}' for i in range(1, 16)],
    'email':       [f'cust{i}@email.com' for i in range(1, 16)],
    'created_date': pd.date_range(start='2019-01-01', periods=15, freq='6ME'),
    'last_activity':pd.date_range(start='2020-06-01', periods=15, freq='4ME'),
    'status':      ['active'] * 8 + ['inactive'] * 4 + ['opted_out'] * 3
})

# Retention policy rules
RETENTION_POLICY = {
    'active':    None,     # Keep indefinitely while active
    'inactive':  365 * 3,  # 3 years since last activity
    'opted_out': 30        # 30 days from opt-out (simulated by last_activity)
}

# TODO: Write a function apply_retention_policy(df, today, policy) that:
#   - For each row, checks if the record exceeds its retention period
#   - Adds a column 'retention_status': 'KEEP', 'ARCHIVE', or 'DELETE'
#   - Adds a column 'days_since_activity'
#   - Returns the annotated DataFrame

def apply_retention_policy(df, today, policy):
    df = df.copy()
    df['days_since_activity'] = (today - df['last_activity']).dt.days

    def get_status(row):
        max_days = policy.get(row['status'])
        if max_days is None:
            return 'KEEP'
        # TODO: return 'DELETE' if days_since_activity > max_days, else 'KEEP'
        pass

    df['retention_status'] = df.apply(get_status, axis=1)
    return df

df_retention = apply_retention_policy(customer_records, today, RETENTION_POLICY)
print(df_retention[['customer_id', 'status', 'last_activity', 'days_since_activity', 'retention_status']].to_string())
print("\nRetention Summary:")
print(df_retention['retention_status'].value_counts())


In [ ]:
# SOLUTION
# import pandas as pd
# from datetime import datetime
#
# today = datetime(2024, 12, 1)
# customer_records = pd.DataFrame({
#     'customer_id': range(1001, 1016),
#     'name':        [f'Customer_{i}' for i in range(1, 16)],
#     'email':       [f'cust{i}@email.com' for i in range(1, 16)],
#     'created_date': pd.date_range(start='2019-01-01', periods=15, freq='6ME'),
#     'last_activity': pd.date_range(start='2020-06-01', periods=15, freq='4ME'),
#     'status': ['active'] * 8 + ['inactive'] * 4 + ['opted_out'] * 3
# })
# RETENTION_POLICY = {'active': None, 'inactive': 365*3, 'opted_out': 30}
#
# def apply_retention_policy(df, today, policy):
#     df = df.copy()
#     df['days_since_activity'] = (today - df['last_activity']).dt.days
#
#     def get_status(row):
#         max_days = policy.get(row['status'])
#         if max_days is None:
#             return 'KEEP'
#         return 'DELETE' if row['days_since_activity'] > max_days else 'KEEP'
#
#     df['retention_status'] = df.apply(get_status, axis=1)
#     return df
#
# df_retention = apply_retention_policy(customer_records, today, RETENTION_POLICY)
# print(df_retention[['customer_id', 'status', 'last_activity', 'days_since_activity', 'retention_status']].to_string())
# print("\nRetention Summary:")
# print(df_retention['retention_status'].value_counts())
# records_to_delete = df_retention[df_retention['retention_status'] == 'DELETE']
# print(f"\nRecords flagged for deletion: {len(records_to_delete)}")


---
## Exercise 5: Data Quality Audit

A **data quality audit** measures the dataset against defined quality dimensions:
- **Completeness** — What % of required fields are populated?
- **Uniqueness** — What % of rows are unique (no duplicates)?
- **Validity** — Do values conform to expected formats/ranges?
- **Accuracy** — Are values correct? (harder to measure automatically)
- **Consistency** — Are values consistent across the dataset?
- **Timeliness** — Is the data current enough for its purpose?

> **Exam Tip:** These are the **six dimensions of data quality** — know all of them by name and definition. The exam frequently presents scenarios asking which dimension is violated: a future date of birth violates *validity*; a missing phone number violates *completeness*; duplicate records violate *uniqueness*.


In [ ]:
import pandas as pd
import numpy as np
import re

# Dataset to audit
audit_data = pd.DataFrame({
    'record_id':  [1, 2, 3, 4, 5, 6, 7, 8, 9, 9],  # 9 is duplicated
    'name':       ['Alice', 'Bob', None, 'Dave', 'Eve', 'Frank', 'Grace', None, 'Ivan', 'Ivan'],
    'email':      ['a@b.com', 'not-an-email', 'c@d.com', None, 'e@f.com', 'g@h.com', 'invalid', 'h@i.com', 'i@j.com', 'i@j.com'],
    'age':        [25, 30, -5, 22, 200, 35, 28, 31, 27, 27],  # -5 and 200 invalid
    'score':      [85, 92, 78, None, 88, 95, 72, 81, None, 79],
    'category':   ['A', 'B', 'C', 'A', 'X', 'B', 'C', 'A', 'B', 'B'],  # 'X' invalid
    'updated_at': pd.to_datetime(['2024-01-15', '2024-02-20', '2023-08-10', '2024-03-01',
                                  '2024-04-05', '2024-01-22', '2024-05-18', '2024-06-30',
                                  '2024-07-14', '2024-07-14'])
})

# TODO: Write audit_data_quality(df) that returns a report dict with:
#   - completeness: % non-null values per column
#   - uniqueness: % unique rows overall (based on record_id)
#   - validity checks:
#       * email: contains '@'
#       * age: between 0 and 120
#       * category: must be in ['A', 'B', 'C']
#   - overall_quality_score: weighted average of the above scores (0-100)

def audit_data_quality(df):
    report = {}

    # TODO: Completeness
    report['completeness'] = {}  # {col: pct_complete}

    # TODO: Uniqueness (based on record_id)
    report['uniqueness_pct'] = None

    # TODO: Validity
    report['validity'] = {}
    # email validity: % with '@'
    # age validity: % in [0, 120]
    # category validity: % in allowed values

    # TODO: Overall score
    report['overall_quality_score'] = None

    return report

quality_report = audit_data_quality(audit_data)
print("=== DATA QUALITY AUDIT REPORT ===")
# TODO: Pretty-print the report


In [ ]:
# SOLUTION
# import pandas as pd
# import numpy as np
# import re
#
# def audit_data_quality(df):
#     report = {}
#     n = len(df)
#
#     # Completeness
#     report['completeness'] = {col: round((df[col].notna().sum() / n) * 100, 1) for col in df.columns}
#     avg_completeness = sum(report['completeness'].values()) / len(report['completeness'])
#
#     # Uniqueness
#     unique_ids = df['record_id'].nunique()
#     report['uniqueness_pct'] = round((unique_ids / n) * 100, 1)
#     report['duplicate_rows'] = n - unique_ids
#
#     # Validity
#     email_valid = df['email'].dropna().apply(lambda x: bool(re.search(r'@', str(x)))).mean() * 100
#     age_valid = df['age'].apply(lambda x: 0 <= x <= 120 if pd.notna(x) else False).mean() * 100
#     cat_valid = df['category'].isin(['A','B','C']).mean() * 100
#     report['validity'] = {
#         'email_format_pct': round(email_valid, 1),
#         'age_range_pct':    round(age_valid, 1),
#         'category_valid_pct': round(cat_valid, 1)
#     }
#     avg_validity = sum(report['validity'].values()) / len(report['validity'])
#
#     report['overall_quality_score'] = round((avg_completeness * 0.4 + report['uniqueness_pct'] * 0.3 + avg_validity * 0.3), 1)
#     return report
#
# quality_report = audit_data_quality(audit_data)
# print("=== DATA QUALITY AUDIT REPORT ===")
# print("\nCompleteness (% non-null):")
# for col, pct in quality_report['completeness'].items():
#     bar = '█' * int(pct // 10) + '░' * (10 - int(pct // 10))
#     print(f"  {col:15s}: {bar} {pct:5.1f}%")
# print(f"\nUniqueness:   {quality_report['uniqueness_pct']:.1f}%  ({quality_report['duplicate_rows']} duplicate rows)")
# print("\nValidity:")
# for check, pct in quality_report['validity'].items():
#     print(f"  {check:30s}: {pct:.1f}%")
# score = quality_report['overall_quality_score']
# grade = 'A' if score >= 90 else 'B' if score >= 80 else 'C' if score >= 70 else 'D' if score >= 60 else 'F'
# print(f"\nOverall Quality Score: {score}/100  (Grade: {grade})")


---
## Exercise 6 (Challenge): Mini Data Governance Audit Report

Combine all previous exercises into a comprehensive governance audit report that:
1. Scans for PII exposure
2. Assesses data quality
3. Checks retention compliance
4. Generates a risk-scored summary

> **Exam Tip:** A **data catalog** is a centralized inventory of all data assets, including metadata (column descriptions, data types, owners, classifications). A **data dictionary** documents the meaning and rules for each field. Both are key governance artifacts the exam tests.


In [ ]:
import pandas as pd
import numpy as np
import re
from datetime import datetime

# Combined dataset for full governance audit
today = datetime(2024, 12, 1)
full_dataset = pd.DataFrame({
    'record_id':    range(1, 21),
    'full_name':    [f'Person {i}' for i in range(1, 21)],
    'email':        [f'p{i}@email.com' if i % 4 != 0 else 'invalid' for i in range(1, 21)],
    'ssn':          [f'{100+i:03d}-{10+i:02d}-{1000+i:04d}' for i in range(1, 21)],
    'age':          [25 + i + (100 if i == 5 else 0) for i in range(20)],  # one outlier
    'salary':       [50000 + i * 2000 for i in range(20)],
    'department':   ['HR', 'IT', 'Finance', 'Marketing'] * 5,
    'created_date': pd.date_range(start='2019-01-01', periods=20, freq='3ME'),
    'last_active':  pd.date_range(start='2020-01-01', periods=20, freq='5ME'),
    'status':       ['active'] * 12 + ['inactive'] * 5 + ['opted_out'] * 3
})

# TODO: Build a comprehensive governance_audit(df, today) function that:
#   1. Detects PII columns → list of PII fields found
#   2. Assesses data quality → completeness %, validity %, uniqueness %
#   3. Applies retention policy → count of records requiring deletion
#   4. Calculates a risk score (0-100, higher = more risk):
#      - +30 if PII columns detected and not masked
#      - +20 if quality score < 80
#      - +30 if any opted_out records exceed 30-day retention
#      - +20 if any records are duplicates
#   5. Prints a formatted governance report with all findings
#   6. Returns the risk score and recommendations list

def governance_audit(df, today):
    print("=" * 60)
    print("       DATA GOVERNANCE AUDIT REPORT")
    print(f"       Generated: {today.strftime('%Y-%m-%d')}")
    print("=" * 60)

    risk_score = 0
    recommendations = []

    # TODO: 1. PII Detection
    print("\n[1] PII EXPOSURE")

    # TODO: 2. Data Quality
    print("\n[2] DATA QUALITY")

    # TODO: 3. Retention Compliance
    print("\n[3] DATA RETENTION")

    # TODO: 4. Duplicate Check
    print("\n[4] UNIQUENESS")

    # TODO: 5. Risk Score
    print("\n[5] RISK ASSESSMENT")
    print(f"    Risk Score: {risk_score}/100")

    print("\n[6] RECOMMENDATIONS")
    for i, rec in enumerate(recommendations, 1):
        print(f"    {i}. {rec}")

    print("\n" + "=" * 60)
    return risk_score, recommendations

risk, recs = governance_audit(full_dataset, today)


In [ ]:
# SOLUTION
# import pandas as pd
# import numpy as np
# import re
# from datetime import datetime
#
# today = datetime(2024, 12, 1)
# full_dataset = pd.DataFrame({
#     'record_id': range(1, 21),
#     'full_name': [f'Person {i}' for i in range(1, 21)],
#     'email': [f'p{i}@email.com' if i % 4 != 0 else 'invalid' for i in range(1, 21)],
#     'ssn': [f'{100+i:03d}-{10+i:02d}-{1000+i:04d}' for i in range(1, 21)],
#     'age': [25 + i + (100 if i == 5 else 0) for i in range(20)],
#     'salary': [50000 + i * 2000 for i in range(20)],
#     'department': ['HR', 'IT', 'Finance', 'Marketing'] * 5,
#     'created_date': pd.date_range(start='2019-01-01', periods=20, freq='3ME'),
#     'last_active': pd.date_range(start='2020-01-01', periods=20, freq='5ME'),
#     'status': ['active'] * 12 + ['inactive'] * 5 + ['opted_out'] * 3
# })
#
# def governance_audit(df, today):
#     print("=" * 60)
#     print("       DATA GOVERNANCE AUDIT REPORT")
#     print(f"       Generated: {today.strftime('%Y-%m-%d')}")
#     print(f"       Dataset: {df.shape[0]} rows, {df.shape[1]} columns")
#     print("=" * 60)
#
#     risk_score = 0
#     recommendations = []
#
#     # 1. PII Detection
#     pii_keywords = ['name', 'email', 'ssn', 'phone', 'address', 'ip']
#     pii_cols_found = [c for c in df.columns if any(kw in c.lower() for kw in pii_keywords)]
#     print("\n[1] PII EXPOSURE")
#     if pii_cols_found:
#         print(f"    ALERT: PII columns detected: {pii_cols_found}")
#         risk_score += 30
#         recommendations.append("Apply data masking to PII columns before sharing with non-privileged roles.")
#     else:
#         print("    OK: No PII columns detected.")
#
#     # 2. Data Quality
#     n = len(df)
#     completeness = df.notna().mean().mean() * 100
#     email_valid_pct = df['email'].apply(lambda x: bool(re.search(r'@', str(x)))).mean() * 100
#     age_valid_pct = df['age'].apply(lambda x: 0 <= x <= 120).mean() * 100
#     quality_score = (completeness + email_valid_pct + age_valid_pct) / 3
#     print("\n[2] DATA QUALITY")
#     print(f"    Completeness:    {completeness:.1f}%")
#     print(f"    Email validity:  {email_valid_pct:.1f}%")
#     print(f"    Age validity:    {age_valid_pct:.1f}%")
#     print(f"    Quality Score:   {quality_score:.1f}/100")
#     if quality_score < 80:
#         risk_score += 20
#         recommendations.append(f"Improve data quality (current score: {quality_score:.1f}%). Fix invalid emails and ages.")
#
#     # 3. Retention
#     df2 = df.copy()
#     df2['days_since_active'] = (today - df2['last_active']).dt.days
#     opted_out_expired = df2[(df2['status'] == 'opted_out') & (df2['days_since_active'] > 30)]
#     inactive_expired = df2[(df2['status'] == 'inactive') & (df2['days_since_active'] > 365*3)]
#     print("\n[3] DATA RETENTION")
#     print(f"    Opted-out records exceeding 30-day policy: {len(opted_out_expired)}")
#     print(f"    Inactive records exceeding 3-year policy:  {len(inactive_expired)}")
#     if len(opted_out_expired) > 0:
#         risk_score += 30
#         recommendations.append(f"Delete {len(opted_out_expired)} opted-out records immediately (GDPR compliance).")
#
#     # 4. Duplicates
#     dup_count = df.duplicated(subset=['record_id']).sum()
#     print("\n[4] UNIQUENESS")
#     print(f"    Duplicate record_ids: {dup_count}")
#     if dup_count > 0:
#         risk_score += 20
#         recommendations.append(f"Remove {dup_count} duplicate records to maintain data integrity.")
#
#     # 5. Risk summary
#     risk_level = 'CRITICAL' if risk_score >= 80 else 'HIGH' if risk_score >= 60 else 'MEDIUM' if risk_score >= 40 else 'LOW'
#     print("\n[5] RISK ASSESSMENT")
#     print(f"    Risk Score: {risk_score}/100  |  Level: {risk_level}")
#
#     print("\n[6] RECOMMENDATIONS")
#     if not recommendations:
#         print("    No critical issues found.")
#     for i, rec in enumerate(recommendations, 1):
#         print(f"    {i}. {rec}")
#
#     print("\n" + "=" * 60)
#     return risk_score, recommendations
#
# risk, recs = governance_audit(full_dataset, today)
